In [2]:
from pathlib import Path
import numpy as np
import imageio.v2 as imageio
from tqdm.auto import tqdm
from astropy.io import fits
from astroquery.mast import Observations

# ---------- USER SETTINGS ----------
TARGET_NAME = "WASP-39"
INSTRUMENT  = "NIRISS"   # "NIRISS" or "NIRSPEC" (case-insensitive)
RADIUS      = "0.2 deg"  # use bigger radius to start

PREFER_SUBGROUPS = ("RATEINTS", "CALINTS")  # best for frame-by-frame video

MAX_OBS_TO_TRY = 50
MAX_FRAMES     = 120
FPS            = 12

OUT_DIR = Path("jwst_tso_video")
DATA_DIR = OUT_DIR / "data"
OUT_DIR.mkdir(exist_ok=True, parents=True)
DATA_DIR.mkdir(exist_ok=True, parents=True)


In [3]:
def _subgroup_col(p):
    for c in p.colnames:
        if c.lower() in ("productsubgroupdescription", "productsubgroupdesc"):
            return c
    return None

def download_one_tso_file(
    target_name: str,
    instrument: str,
    prefer=("RATEINTS", "CALINTS"),
    radius="0.2 deg",
    download_dir=Path("."),
    max_obs_to_try=50,
    verbose=True
) -> Path:
    # 1) Search by coordinates of the target
    obs = Observations.query_object(target_name, radius=radius)

    # 2) Keep JWST only
    obs = obs[obs["obs_collection"] == "JWST"]
    if len(obs) == 0:
        raise RuntimeError(f"No JWST observations found for '{target_name}' (radius={radius}).")

    # 3) Instrument filter MUST be "contains", not equality
    inst = obs["instrument_name"].astype(str)
    instrument = instrument.upper()
    keep = np.char.find(np.char.upper(inst), instrument) >= 0
    obs = obs[keep]

    if len(obs) == 0:
        # Helpful diagnostics
        raise RuntimeError(
            f"No JWST observations matched instrument contains '{instrument}' for '{target_name}'.\n"
            f"Tip: try INSTRUMENT='NIRSPEC' or increase RADIUS. "
        )

    if verbose:
        print("Found candidate observations:", len(obs))
        print("Unique instrument_name values (first 15):")
        for x in np.unique(obs["instrument_name"].astype(str))[:15]:
            print("  ", x)

    # 4) Try observations until we find RATEINTS/CALINTS product
    obs = obs[:max_obs_to_try]

    for i, o in enumerate(obs):
        if verbose:
            print(f"\nTrying obs {i+1}/{len(obs)}  obs_id={o['obs_id']}  instrument_name={o['instrument_name']}")

        products = Observations.get_product_list(o)

        # Keep SCIENCE FITS
        p = Observations.filter_products(products, productType="SCIENCE", extension="fits")
        if len(p) == 0:
            if verbose:
                print("  No SCIENCE FITS products.")
            continue

        subgroup_col = _subgroup_col(p)
        fn = p["productFilename"].astype(str)

        candidates = None
        for key in prefer:
            # by subgroup label
            if subgroup_col and np.any(p[subgroup_col] == key):
                candidates = p[p[subgroup_col] == key]
                if verbose:
                    print(f"  Found subgroup={key}: {len(candidates)} files")
                break

            # fallback by filename match
            if np.any(np.char.find(np.char.lower(fn), key.lower()) >= 0):
                candidates = p[np.char.find(np.char.lower(fn), key.lower()) >= 0]
                if verbose:
                    print(f"  Found filename contains '{key.lower()}': {len(candidates)} files")
                break

        if candidates is None or len(candidates) == 0:
            if verbose:
                print("  No RATEINTS/CALINTS here.")
            continue

        # Download just the first candidate (simple)
        one = candidates[0:1]
        manifest = Observations.download_products(one, download_dir=str(download_dir), cache=True)

        local_col = "Local Path" if "Local Path" in manifest.colnames else "local_path"
        fits_path = Path(manifest[local_col][0])

        if fits_path.exists():
            if verbose:
                print("  Downloaded:", fits_path)
            return fits_path

    raise RuntimeError(
        "Searched observations but did not find RATEINTS/CALINTS.\n"
        "Try switching INSTRUMENT ('NIRSPEC' vs 'NIRISS'), changing TARGET_NAME, or increasing MAX_OBS_TO_TRY."
    )

fits_path = download_one_tso_file(
    TARGET_NAME,
    instrument=INSTRUMENT,
    prefer=PREFER_SUBGROUPS,
    radius=RADIUS,
    download_dir=DATA_DIR,
    max_obs_to_try=MAX_OBS_TO_TRY,
)
print("\nUsing FITS:", fits_path)


Found candidate observations: 2
Unique instrument_name values (first 15):
   NIRISS/SOSS

Trying obs 1/2  obs_id=jw01366-o001_t001_niriss_clear-gr700xd-substrip256  instrument_name=NIRISS/SOSS
  Found subgroup=RATEINTS: 8 files
  Downloaded: jwst_tso_video/data/mastDownload/JWST/jw01366001001_02101_00004-seg001_nis/jw01366001001_02101_00004-seg001_nis_rateints.fits

Using FITS: jwst_tso_video/data/mastDownload/JWST/jw01366001001_02101_00004-seg001_nis/jw01366001001_02101_00004-seg001_nis_rateints.fits


In [4]:
def to_frame_cube(arr: np.ndarray) -> np.ndarray:
    """
    Convert common JWST shapes to (nframes, y, x).
    2D -> 1 frame
    3D -> assume (nints, y, x) OR (y, x, nints)
    4D -> assume (nints, ngroups, y, x) and average over groups
    """
    a = np.asarray(arr)
    if a.ndim == 2:
        return a[None, :, :]
    if a.ndim == 3:
        # If first axis looks like time/integrations, use it.
        if a.shape[0] <= 5000:
            return a
        # Else if last axis looks like time, move it.
        if a.shape[2] <= 5000:
            return np.moveaxis(a, 2, 0)
        raise ValueError(f"Unclear 3D layout: {a.shape}")
    if a.ndim == 4:
        # Average over groups -> per integration frames
        return np.nanmean(a, axis=1)
    raise ValueError(f"Unsupported ndim={a.ndim}, shape={a.shape}")

with fits.open(fits_path) as hdul:
    hdul.info()
    if "SCI" not in hdul:
        raise RuntimeError("No SCI extension. Check hdul.info() and adjust.")
    sci = hdul["SCI"].data

frames = to_frame_cube(sci).astype(np.float32)
print("Frame cube:", frames.shape)  # (nframes, y, x)


Filename: jwst_tso_video/data/mastDownload/JWST/jw01366001001_02101_00004-seg001_nis/jw01366001001_02101_00004-seg001_nis_rateints.fits
No.    Name      Ver    Type      Cards   Dimensions   Format
  0  PRIMARY       1 PrimaryHDU     214   ()      
  1  SCI           1 ImageHDU        76   (64, 64, 1)   float32   
  2  ERR           1 ImageHDU        11   (64, 64, 1)   float32   
  3  DQ            1 ImageHDU        12   (64, 64, 1)   int32 (rescales to uint32)   
  4  INT_TIMES     1 BinTableHDU     24   1R x 7C   [J, D, D, D, D, D, D]   
  5  VAR_POISSON    1 ImageHDU        10   (64, 64, 1)   float32   
  6  VAR_RNOISE    1 ImageHDU        10   (64, 64, 1)   float32   
  7  ASDF          1 BinTableHDU     11   1R x 1C   [17872B]   
Frame cube: (1, 64, 64)


In [5]:
def scale_to_uint8(img2d, vmin, vmax, stretch="asinh"):
    x = np.nan_to_num(img2d, nan=vmin)
    x = np.clip((x - vmin) / (vmax - vmin + 1e-12), 0, 1)

    if stretch == "asinh":
        x = np.arcsinh(10 * x) / np.arcsinh(10)
    elif stretch == "sqrt":
        x = np.sqrt(x)
    elif stretch == "log":
        x = np.log1p(1000 * x) / np.log1p(1000)

    return (255 * x).astype(np.uint8)

n = min(len(frames), MAX_FRAMES)
use = frames[:n]

# Global scaling prevents flicker
vmin, vmax = np.nanpercentile(use, [1, 99.7])
print("vmin/vmax:", vmin, vmax)


vmin/vmax: -26.42080726623535 1200.0960519409614


In [6]:
video_path = OUT_DIR / f"{TARGET_NAME.replace(' ','_')}_{INSTRUMENT}_tso.mp4"
print("Writing:", video_path)

with imageio.get_writer(video_path, fps=FPS, codec="libx264", quality=8) as w:
    for i in tqdm(range(n), desc="Encoding"):
        u8 = scale_to_uint8(use[i], vmin, vmax, stretch="asinh")
        rgb = np.repeat(u8[..., None], 3, axis=2)
        w.append_data(rgb)

print("Done:", video_path.resolve())


Writing: jwst_tso_video/WASP-39_NIRISS_tso.mp4


Encoding:   0%|          | 0/1 [00:00<?, ?it/s]

Done: /home/hw1970218/Desktop/fits2/jwst_tso_video/WASP-39_NIRISS_tso.mp4


In [7]:
from IPython.display import Video, display
display(Video(str(video_path), embed=True))
